**Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Is the stock market overvalued? Part 1: getting and cleaning data

So far, we've found data, cleaned it in Excel, uploaded it, and charted it, all by hand. Now we'll do the same steps **in code**, so they can be repeated and checked.

**By the end you will have:**
1. Fetched files from the internet with Python
2. Calculated a new column (the price-to-earnings ratio)
3. Removed missing data
4. Fixed the dates to `yyyy-mm-dd`, and caught a sneaky bug
5. Saved a clean CSV, ready to chart

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.
>
> Run a cell by clicking it and pressing **Shift + Enter**.

## What is a notebook?

A notebook mixes two kinds of cell:
- **Text cells** (like this one): explanations, headings and links
- **Code cells**: Python you can run, with the result shown underneath

We run code in small steps and check each result before moving on. We use **Google Colab**, which runs Python in your browser with nothing to install.

## 0. Load our packages

In [ ]:
import requests        # fetch files from the internet
import pandas as pd     # work with tables of data

---

## 1. Fetching from the internet with `requests`

`requests.get(url)` downloads whatever is at a web address. Let's fetch a Vega-Lite chart from Richard's chart library.

In [ ]:
chart_url = "https://raw.githubusercontent.com/RDeconomist/RDeconomist.github.io/refs/heads/main/charts/library/chartLine3.json"

chart_response = requests.get(chart_url)
chart_response

`<Response [200]>` means it worked (200 is the web's code for "OK").

The file is JSON, so `.json()` turns it into a Python **dictionary**:

In [ ]:
chart_spec = chart_response.json()
chart_spec

This is the same Vega-Lite JSON you've been editing. Because it's now a dictionary, we can read parts of it with keys:

In [ ]:
chart_spec["title"]

> **Why this matters:** many data sources (APIs) send back JSON like this. We'll use exactly this technique to fetch live data in Week 4.

---

## 2. Loading a CSV into a table

Now, some data. We'll use Robert Shiller's long-run data on US stock prices and company earnings, starting in 1871 ([shillerdata.com](https://shillerdata.com/)). A copy is stored as a CSV on GitHub.

For CSV files, `pandas` can read straight from the raw GitHub link:

In [ ]:
csv_url = "https://raw.githubusercontent.com/jhellingsdata/jhellingsdata.github.io/refs/heads/main/Data/data_shiller.csv"

df = pd.read_csv(csv_url)

`df` (short for **D**ata**F**rame) is now a table, like a sheet in Excel. `.head()` shows the first 5 rows:

In [ ]:
df.head()

A few useful checks on any new dataset:

In [ ]:
df.shape      # (rows, columns)

In [ ]:
df.columns    # the column names

The columns we need:

| Column | Meaning |
|---|---|
| `Date` | Year and month, written as `1871.01` |
| `P` | Stock price index (S&P composite) |
| `E` | Company earnings |

Pick out one column with `df['column name']`:

In [ ]:
df['P']

<details>
<summary><b>Optional: loading a CSV with <code>requests</code> instead</b></summary>

`pd.read_csv(url)` works for simple links like raw GitHub files. For some websites you need to fetch the file first with `requests`, then read it:

```python
import io
response = requests.get(csv_url)
df = pd.read_csv(io.StringIO(response.text))
```
</details>

---

## 3. Calculating a new column

The **price-to-earnings (P/E) ratio** is the stock price divided by company earnings. A high P/E means investors are paying a lot for each £1 (or $1) of profit, which can be a sign the market is overvalued.

Python does maths on a whole column at once:

In [ ]:
df['P'] / df['E']

To keep the result, store it as a new column. Choose the name in the square brackets:

In [ ]:
df['PE_ratio'] = df['P'] / df['E']

df.tail()   # .tail() shows the last 5 rows

---

## 4. Missing data

The last rows show `NaN` ("not a number"). That means **missing**: the most recent earnings haven't been published yet, so there's no P/E to calculate.

How many are missing?

In [ ]:
df['PE_ratio'].isna().sum()

`.dropna()` removes rows with missing values. `subset=` says which column to check.

We store the result as a new table, `df_clean`, so the original `df` is still there if we need it. (`.copy()` tells pandas this is a separate table.)

In [ ]:
df_clean = df.dropna(subset=['PE_ratio']).copy()

df_clean.tail()

In [ ]:
# Check: no missing values left
df_clean['PE_ratio'].isna().sum()

---

## 5. Fixing the dates

Vega-Lite wants dates as `yyyy-mm-dd`. Ours look like `1871.01` (year.month).

`pd.to_datetime()` converts text to dates. We tell it the current pattern with `format=`:
- `%Y` is a 4-digit year
- `%m` is a 2-digit month

(These are the same date codes Vega-Lite uses.)

In [ ]:
df_clean['Date_iso'] = pd.to_datetime(df_clean['Date'].astype(str), format='%Y.%m')

df_clean[['Date', 'Date_iso']].head()

Looks good! But **always check**. Monthly data should have exactly one row per month, so no date should appear twice:

In [ ]:
df_clean['Date_iso'].duplicated().sum()

### 🔍 Something's wrong

Over 150 duplicate dates! Let's look at the end of 1871:

In [ ]:
df_clean[['Date', 'Date_iso']].iloc[8:12]

**October became January.** Why?

The CSV stores the year and month as a **number**. As a number, `1871.10` is exactly the same as `1871.1`, so the trailing zero is dropped. When the date is read, `.1` is taken to mean month **1**.

This kind of bug is easy to miss: no error, and the chart still draws. Only checking the data catches it.

**The fix:** turn each number into text with exactly 2 decimal places, so `1871.1` becomes `'1871.10'`, before reading it as a date.

In [ ]:
df_clean['Date'] = df_clean['Date'].map('{:.2f}'.format)   # 1871.1 -> '1871.10'

df_clean['Date_iso'] = pd.to_datetime(df_clean['Date'], format='%Y.%m')

df_clean[['Date', 'Date_iso']].iloc[8:12]

In [ ]:
# Check again: should be 0
df_clean['Date_iso'].duplicated().sum()

---

## 6. Save a clean CSV

We only need two columns for our chart. Keeping just these makes the file smaller and easier to use.

In [ ]:
df_out = df_clean[['Date_iso', 'PE_ratio']]

df_out.to_csv('data_shiller_pe.csv', index=False)   # index=False: don't save pandas' row numbers

df_out.head()

The file is saved in Colab's **Files** panel (the folder icon on the left).

> **Colab files are temporary.** They are deleted when your session ends, so download the file now:

In [ ]:
try:
    from google.colab import files     # only exists in Colab
    files.download('data_shiller_pe.csv')
except ImportError:
    print('Not running in Colab: the file is saved next to this notebook.')

### Next steps

1. Upload `data_shiller_pe.csv` to your GitHub repository (just like in Week 2).
2. Open it on GitHub, click **Raw**, and copy the link.

In the seminar, we'll use that link to chart the data with Python (**DS3_2_Altair_Shiller**).

---

## <font color='green'>Stretch exercises</font>

**1.** What was the highest P/E ratio, and when? *Hint:* `.max()`, and `df_clean.sort_values('PE_ratio')`.

In [ ]:
# Your code here


**2.** Calculate the **average** P/E ratio over the whole period with `.mean()`. Is today's value above or below it?

In [ ]:
# Your code here


**3.** The data also has `D` (dividends). Calculate a **dividend yield** column: `D / P * 100`.

In [ ]:
# Your code here
